# RLVR TerminalBench Training on Colab

Train a larger model (Qwen2.5-3B-Instruct) with GRPO on a free T4 GPU (15GB VRAM).

**Runtime:** Go to Runtime > Change runtime type > Select **T4 GPU**

## 1. Setup

In [1]:
# Check GPU
!nvidia-smi

Fri Mar 27 18:03:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install dependencies
!pip install -q transformers trl accelerate datasets peft bitsandbytes pyyaml tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00


In [3]:
# Clone the repo (or upload files manually)
!git clone https://github.com/NathanG2022/rlvr-terminalbench.git 2>/dev/null || echo 'Repo already cloned'

# If repo is private or not on GitHub, upload the project files instead:
# from google.colab import files
# files.upload()  # upload a zip, then unzip

import os
os.chdir('rlvr-terminalbench')
print('Working directory:', os.getcwd())

Working directory: /content/rlvr-terminalbench


## 2. Upload project files (if not using git clone)

If your repo isn't on GitHub, run this cell and upload a zip of the project:

In [4]:
# Alternative: upload zip file
# from google.colab import files
# uploaded = files.upload()
# !unzip -o rlvr-terminalbench.zip
# os.chdir('rlvr-terminalbench')

## 3. Verify GPU + imports

In [5]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found! Go to Runtime > Change runtime type > T4 GPU')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 4. Configure model and training

In [6]:
# Model choice - Qwen2.5-3B is free, no approval needed, 3x bigger than TinyLlama
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # smaller, for quick tests
# MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"  # needs HF approval + login

# Training params
NUM_PROMPTS = 128       # number of task prompts in dataset
TOTAL_STEPS = 200       # training steps (increase for better results)
BATCH_SIZE = 4          # prompts per batch
NUM_GENERATIONS = 4     # completions per prompt (more = better GRPO signal)
LEARNING_RATE = 5e-6
MAX_NEW_TOKENS = 64
MAX_STEPS_PER_EPISODE = 5

print(f'Model: {MODEL_NAME}')
print(f'Steps: {TOTAL_STEPS}, Batch: {BATCH_SIZE}, Generations: {NUM_GENERATIONS}')

Model: Qwen/Qwen2.5-3B-Instruct
Steps: 200, Batch: 4, Generations: 4


## 5. Build prompt dataset

In [7]:
import sys
sys.path.insert(0, '.')

from datasets import Dataset
from envs.terminalbench_client import TerminalBenchClient
from envs.terminalbench_env import TerminalBenchEnv

tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
env = TerminalBenchEnv(
    tb_client,
    max_steps=MAX_STEPS_PER_EPISODE,
    step_penalty=0.01,
    w_success=1.0,
    w_eff=0.1,
    w_quality=0.05,
)

prompts = []
for _ in range(NUM_PROMPTS):
    obs = env.reset()
    prompts.append({'prompt': obs})

train_dataset = Dataset.from_list(prompts)
print(f'Dataset: {len(train_dataset)} prompts')
print(f'\nExample prompt:\n{prompts[0]["prompt"][:300]}...')

Dataset: 128 prompts

Example prompt:
[terminalbench]
Task: Write the first 3 lines of 'data.txt' to a new file called 'top.txt'.
Difficulty: easy
Step: 0/5

Terminal history:

You are a CLI assistant. Issue the next bash command to complete the task.
Command:...


## 6. Define reward function

In [8]:
import re

def extract_command(text: str) -> str:
    """Extract a single shell command from model output."""
    text = text.strip()
    if not text:
        return 'echo noop'

    m = re.search(r'```(?:bash|sh)?\s*\n(.+?)```', text, re.DOTALL)
    if m:
        for line in m.group(1).splitlines():
            line = line.strip()
            if line and not line.startswith('#'):
                return line

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if re.match(r'^[A-Z][a-z].*\s(the|a|an|is|to|you|can|this)\s', line):
            continue
        return line

    return text.splitlines()[0].strip() or 'echo noop'


def make_reward_func():
    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client,
        max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=0.01,
        w_success=1.0,
        w_eff=0.1,
        w_quality=0.05,
    )

    def reward_func(prompts, completions, **kwargs):
        rewards = []
        for prompt, completion in zip(prompts, completions):
            command = extract_command(completion)
            env.reset()
            _obs, reward, _done, _info = env.step(command)
            rewards.append(reward)
        return rewards

    return reward_func

reward_func = make_reward_func()
print('Reward function ready')

Reward function ready


## 7. Load model with 4-bit quantization + LoRA

In [9]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
)

# Check memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory: {mem_used:.1f} / {mem_total:.1f} GB')

print('Model loaded')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU memory: 2.2 / 15.6 GB
Model loaded


## 8. Train with GRPO

In [10]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = 'models/rlvr_colab'

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    num_train_epochs=1,
    max_steps=TOTAL_STEPS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.7,
    top_p=0.9,
    beta=0.04,
    logging_steps=10,
    save_strategy='steps',
    save_steps=100,
    report_to='none',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    args=grpo_config,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

print(f'Training for {TOTAL_STEPS} steps...')
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training for 200 steps...


Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
10,0.000027
20,0.000096
30,0.000082
40,0.000121
50,0.000212
60,0.000064
70,0.000080
80,0.000088
90,0.000085
100,0.000078


Model saved to models/rlvr_colab


## 9. Evaluate trained model

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from statistics import mean

def evaluate_model(model_path, num_episodes=20):
    """Evaluate a model on terminalbench tasks."""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    eval_model = AutoModelForCausalLM.from_pretrained(model_path).eval().cuda()
    max_ctx = getattr(eval_model.config, 'max_position_embeddings', 2048)

    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client, max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=0.01, w_success=1.0, w_eff=0.1, w_quality=0.05,
    )

    scores, steps = [], []
    for ep in range(num_episodes):
        obs = env.reset()
        done = False
        task_id = env.task.task_id if env.task else 'unknown'
        last_score = 0.0

        while not done:
            max_new = MAX_NEW_TOKENS
            inputs = tokenizer(
                obs, return_tensors='pt', truncation=True,
                max_length=max_ctx - max_new,
            ).to(eval_model.device)
            out_ids = eval_model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=max_new,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_ids = out_ids[0][inputs['input_ids'].shape[1]:]
            action = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            action = action.split('\n')[0].strip().lstrip('$ ').strip()
            print(f'  ep{ep+1} step{env.step_count+1}: {action[:80]}')
            obs, _reward, done, info = env.step(action)
            last_score = info['success_score']

        scores.append(last_score)
        steps.append(info['step_count'])
        print(f'  -> task={task_id} score={last_score:.2f} steps={info["step_count"]}')

    # Free GPU memory
    del eval_model
    torch.cuda.empty_cache()

    return {
        'mean_score': mean(scores),
        'success_rate': mean(1.0 if s >= 1.0 else 0.0 for s in scores),
        'mean_steps': mean(steps),
    }

In [12]:
NUM_EVAL_EPISODES = 20

print('=== Evaluating TRAINED model ===')
trained_results = evaluate_model(OUTPUT_DIR, num_episodes=NUM_EVAL_EPISODES)

print(f'\n=== Evaluating BASE model ({MODEL_NAME}) ===')
base_results = evaluate_model(MODEL_NAME, num_episodes=NUM_EVAL_EPISODES)

print('\n' + '='*50)
print(f'{"Metric":<25} {"Base":>10} {"Trained":>10}')
print('-'*50)
print(f'{"Mean score":<25} {base_results["mean_score"]:>10.3f} {trained_results["mean_score"]:>10.3f}')
print(f'{"Success rate":<25} {base_results["success_rate"]:>10.3f} {trained_results["success_rate"]:>10.3f}')
print(f'{"Mean steps":<25} {base_results["mean_steps"]:>10.2f} {trained_results["mean_steps"]:>10.2f}')
print('='*50)

=== Evaluating TRAINED model ===


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

  ep1 step1: `rm trash.txt` To delete the file `trash.txt`, you can use the `rm` (remove) com
  ep1 step2: `/usr/bin/rm /tmp/tb_easy_delete_file_kxjzivch/trash.txt` Step: 2/5
  -> task=easy_delete_file score=1.00 steps=2
  ep2 step1: ```bash
  ep2 step2: ```bash
  ep2 step3: python -c 'import json, sys; x=json.load(open("config.json")); print(x["database
  -> task=hard_json_extract score=1.00 steps=3
  ep3 step1: cat words.txt | tr ' ' '\n' | sort | uniq | wc -l > count.txt
  -> task=med_unique_words score=1.00 steps=1
  ep4 step1: ```
  ep4 step2: ```
  ep4 step3: rm trash.txt
  -> task=easy_delete_file score=1.00 steps=3
  ep5 step1: cat data.txt | wc -l > count.txt
  -> task=easy_count_lines score=1.00 steps=1
  ep6 step1: `cat a.txt > merged.txt && cat b.txt >> merged.txt`
  -> task=med_merge_files score=1.00 steps=1
  ep7 step1: echo "hello world" > hello.txt
  -> task=easy_create_hello score=1.00 steps=1
  ep8 step1: `tar -czvf archive.tar.gz project/` To create a tar.gz archive 

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  ep1 step1: touch hello.txt
  ep1 step2: echo "hello world" > hello.txt
  -> task=easy_create_hello score=1.00 steps=2
  ep2 step1: `cat words.txt | tr ' ' '\n' | sort | uniq -c | sort -nr | awk '{print $2 " " $1
  -> task=med_word_freq score=1.00 steps=1
  ep3 step1: `mv old.txt new.txt` To rename the file `old.txt` to `new.txt`, you can use the 
  -> task=easy_rename_file score=1.00 steps=1
  ep4 step1: touch empty.txt
  -> task=easy_create_empty score=1.00 steps=1
  ep5 step1: `mv old.txt new.txt` To rename the file `old.txt` to `new.txt`, you can use the 
  -> task=easy_rename_file score=1.00 steps=1
  ep6 step1: cat a.txt b.txt > merged.txt
  -> task=med_merge_files score=1.00 steps=1
  ep7 step1: `echo hello > output.txt`
  ep7 step2: ```bash
  ep7 step3: ```bash
  ep7 step4: ```
  ep7 step5: ```
  -> task=hard_shell_script score=0.00 steps=5
  ep8 step1: cp source.txt dest.txt
  -> task=easy_copy_file score=1.00 steps=1
  ep9 step1: `tail -5 log.txt > last5.txt`
  -> task=med_t

## 10. Download trained model

In [13]:
# Zip and download the trained model
!zip -r models/rlvr_colab.zip models/rlvr_colab/

from google.colab import files
files.download('models/rlvr_colab.zip')

  adding: models/rlvr_colab/ (stored 0%)
  adding: models/rlvr_colab/README.md (deflated 47%)
  adding: models/rlvr_colab/tokenizer_config.json (deflated 59%)
  adding: models/rlvr_colab/checkpoint-200/ (stored 0%)
  adding: models/rlvr_colab/checkpoint-200/trainer_state.json (deflated 87%)
  adding: models/rlvr_colab/checkpoint-200/optimizer.pt (deflated 22%)
  adding: models/rlvr_colab/checkpoint-200/README.md (deflated 66%)
  adding: models/rlvr_colab/checkpoint-200/tokenizer_config.json (deflated 59%)
  adding: models/rlvr_colab/checkpoint-200/scheduler.pt (deflated 62%)
  adding: models/rlvr_colab/checkpoint-200/training_args.bin (deflated 53%)
  adding: models/rlvr_colab/checkpoint-200/chat_template.jinja (deflated 71%)
  adding: models/rlvr_colab/checkpoint-200/adapter_config.json (deflated 57%)
  adding: models/rlvr_colab/checkpoint-200/adapter_model.safetensors (deflated 21%)
  adding: models/rlvr_colab/checkpoint-200/tokenizer.json (deflated 81%)
  adding: models/rlvr_colab/c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>